<a href="https://colab.research.google.com/github/Bharathyramakrishnan/Applied-Generative-AI-Specialization/blob/main/AutoEncoders.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **AutoEncoder Using Keras on Mnist DataSet .**


In [ ]:
import matplotlib.pyplot as plt # used for displaying images and plots
from keras.datasets import mnist # sample dataset contains handwritten numbers as images
from keras.layers import Input, Dense # Input layer is shape of the layer like 28*28 and Dense is layers where all the neurons are connected to the next layer of neurons
from keras.models import Model

In [ ]:
# Loading The Data
(x_train, _), (x_test, _) = mnist.load_data() # X_train is original image, y_trian we give as what is the image means

In [ ]:
# PreProcess The Data
def pre_process(X):
    X = X/255.0   # Normally in images Satuartion of colors range from 0 to 255, x/255 means the saturation of every pixel divided by 255
    X = X.reshape((len(X), 784))
    return X


# 0 means no saturation of colors and 255 means full saturation of color,
# We want to make sure that the saturation of color ranges from 0 to one. This a common practice.
#This allows your values not to be super big, but also to remain between 0 and one. So that's the reason why you see here X.

    """
    Preprocesses image data by normalizing pixel values and flattening the input.

    This function is designed to prepare grayscale image data (e.g., MNIST) for input
    into a machine learning model. It performs the following preprocessing steps:

    1. Normalization: Divides all pixel values by 255.0 to scale them from the
       original [0, 255] range to the [0, 1] range. This helps improve training
       stability and convergence in neural networks.

    2. Flattening: Reshapes each 2D image (typically 28x28) into a 1D vector of
       length 784. The entire dataset is then reshaped to a 2D array of shape
       (num_samples, 784), where each row represents a flattened image.

    Parameters:
    -----------
    X : numpy.ndarray
        A 3D NumPy array of shape (num_samples, height, width), where each element
        represents a grayscale image with pixel values ranging from 0 to 255.

    Returns:
    --------
    numpy.ndarray
        A 2D NumPy array of shape (num_samples, 784), where each row is a normalized
        and flattened representation of the original image.

    Example:
    --------
    >>> X.shape
    (60000, 28, 28)
    >>> X_preprocessed = pre_process(X)
    >>> X_preprocessed.shape
    (60000, 784)
    """

In [ ]:
x_train  =  pre_process(x_train)
x_test  =  pre_process(x_test)

In [ ]:
# Function to Display The Data
def display_data(data , height, width, title):
    plt.figure(figsize=(10, 2))
    for i in range(10):
        ax = plt.subplot(1,10,i+1)
        plt.imshow(data[i].reshape((height,width)))
        plt.gray()
    plt.suptitle(title)

In [ ]:
def AutoEncoder():

    # Input Layer
    input_layer = Input(shape=(784,), name="input_layer") # shape 784 which is 28*28

    # encoder that maps the input into the code (compressed representation).
    h_1 = Dense(256, activation='relu', name="hidden_layer_1")(input_layer) # first layer connected to input layer shape reduced to 256 and so on
    h_2 = Dense(128, activation='relu', name="hidden_layer_2")(h_1) # second layer connect to first
    h_3 = Dense(64, activation='relu', name="hidden_layer_3")(h_2) # 3rd to second
    h_4 = Dense(32, activation='relu', name="hidden_layer_4")(h_3) # 4th to 3rd
    h_5 = Dense(16, activation='relu', name="hidden_layer_5")(h_4) # 5th to 4th
    h_6 = Dense(8, activation='relu', name="hidden_layer_6")(h_5) # 6th to 5th

    # code layer
    code_layer = Dense(4, activation='relu', name="code_layer")(h_6) # related representation of original data or image have only 4 neurons

    # decoder that maps the code to a reconstruction of the input.
    h_7 = Dense(8, activation='relu', name="hidden_layer_7")(code_layer) # first layer of decoder with code layer as input and the steps repeted until we get the output layer
    h_8 = Dense(16, activation='relu', name="hidden_layer_8")(h_7)
    h_9 = Dense(32, activation='relu', name="hidden_layer_9")(h_8)
    h_10 = Dense(64, activation='relu', name="hidden_layer_10")(h_9)
    h_11 = Dense(128, activation='relu', name="hidden_layer_11")(h_10)
    h_12 = Dense(256, activation='relu', name="hidden_layer_12")(h_11)

    # Output Layer
    output_layer = Dense(784, activation='sigmoid', name="output_layer")(h_12) # last layer with sigmoid

    model = Model(input_layer, output_layer) # we got the model output
    model.compile(optimizer='adam', loss='binary_crossentropy')
    model.summary()

    return model

![image.png]()

In [ ]:
# Training The Model
Auto_Encoder = AutoEncoder()
Auto_Encoder.fit(x_train, x_train, epochs=20)

# Get The Decoded Data (The Final Output)
decoded_data = Auto_Encoder.predict(x_test)

# Get The Encoder Part of The Model
encoder = Model(inputs=Auto_Encoder.input, outputs=Auto_Encoder.get_layer("code_layer").output)

# get The Encoded Data (compressed representation of size 2*2)
encoded_data = encoder.predict(x_test)

In [ ]:
# Display The Final Result
display_data(x_test, height=28, width=28 , title="original data")
#display_data(encoded_data, height=8, width=8, title="encoded data")
display_data(decoded_data, height=28, width=28 , title="decoded data")

https://medium.com/@AnasBrital98/autoencoders-explained-da131e60e02a

# **AutoEncoder Using PyTorch on Mnist DataSet**

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

In [ ]:
# Loading The DataSet
train_data = datasets.MNIST(
    root = 'data',
    train = True,
    transform = transforms.ToTensor(),
    download = True,
)


In [ ]:
train_data_loader = torch.utils.data.DataLoader(dataset=train_data,batch_size=64,shuffle=True)

In [ ]:
# Defining our AutoEncoder Architecture
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder , self).__init__()

        # Encoder
        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 32),
            nn.ReLU(),
            nn.Linear(32, 16) ,
            nn.ReLU(),
            nn.Linear(16, 10)
        )

        # Decoder
        self.decoder = nn.Sequential(
            nn.Linear(10, 16),
            nn.ReLU(),
            nn.Linear(16, 32),
            nn.ReLU(),
            nn.Linear(32, 64),
            nn.ReLU(),
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded_data = self.encoder(x)
        decoded_data = self.decoder(encoded_data)
        return decoded_data



In [ ]:
# Training The Model

model = Autoencoder()
loss_function = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3,weight_decay=1e-5)

number_of_epochs = 10
results = []
for epoch in range(number_of_epochs):
    for (img, _) in train_data_loader:
        img = img.reshape(-1 , 784)
        reconstructed_image = model(img)
        loss = loss_function(reconstructed_image, img)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

    print(f'Epoch : {epoch+1} , Loss : {loss.item()} ')
    results.append((epoch, img, reconstructed_image))

In [ ]:
# Visualize The Result

def plot_result(autoEncoder_result , nbr_epoch):
    original_images = autoEncoder_result[ nbr_epoch -1 ][1].detach().numpy()
    reconstructed_images = autoEncoder_result[ nbr_epoch -1 ][2].detach().numpy()

    # Plot The Original Images
    plt.figure(figsize=(9, 3))
    plt.gray()
    for index, image in zip(range(9) , original_images):
            plt.subplot(2, 9, index+1)
            image = image.reshape(-1, 28,28)
            plt.imshow(image[0])
    plt.suptitle(f"Original Images " , fontsize = 20)
     # Plot The Reconstructed images
    plt.figure(figsize=(9, 3))
    plt.gray()
    for index, image in zip(range(9), reconstructed_images):
        plt.subplot(2, 9, index+1)
        image = image.reshape(-1, 28,28)
        plt.imshow(image[0])
    plt.suptitle(" Reconstructed Images" , fontsize = 20)

In [ ]:


plot_result(results , number_of_epochs)